# Diverse Models (Out-of-Fold)
This notebook trains diverse machine learning models (e.g., Random Forest, Extra Trees, Naive Bayes) to generate out-of-fold (OOF) predictions for the ensembling stage.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
import optuna
import warnings

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
print("Loading data...")
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

In [ ]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [ ]:
# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

In [ ]:
def apply_mappings(df):
    df_out = df.copy()
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    df_out['gender'] = df_out['gender'].fillna('Unknown')
    return df_out

In [ ]:
X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

In [ ]:
# Feature Engineering
def add_features(df):
    df_out = df.copy()
    denom_screen = df_out['daily_screen_time_hours'].replace(0, 0.001)
    denom_notif = df_out['notifications_per_day'].replace(0, 0.001)

    df_out['social_media_ratio'] = df_out['social_media_hours'] / denom_screen
    df_out['gaming_ratio'] = df_out['gaming_hours'] / denom_screen
    df_out['work_study_ratio'] = df_out['work_study_hours'] / denom_screen
    df_out['app_opens_per_hour'] = df_out['app_opens_per_day'] / denom_screen
    df_out['notifications_to_opens_ratio'] = df_out['app_opens_per_day'] / denom_notif
    df_out['sleep_deficit'] = 8.0 - df_out['sleep_hours']
    return df_out

In [ ]:
X_preprocessed = add_features(X_preprocessed)
X_test_preprocessed = add_features(X_test_preprocessed)

In [ ]:
# Columns
categorical_cols = ['gender']
numeric_cols = [col for col in X_preprocessed.columns if col not in categorical_cols]

In [ ]:
# Advanced Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', QuantileTransformer(output_distribution='normal', random_state=42))
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
# Constants
N_FOLDS = 5
N_TRIALS = 10
cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
def eval_pipeline(model, X_train, y_train):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = []
    for train_idx, val_idx in cv_tune.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        pipeline.fit(X_tr, y_tr)
        preds = pipeline.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
    return np.mean(scores)

### Optuna Tuning

In [ ]:
print("--- Starting Optuna Tuning ---")

In [ ]:
# 1. ExtraTreesClassifier
def objective_et(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 300)
    max_depth = trial.suggest_int('max_depth', 5, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    model = ExtraTreesClassifier(n_estimators=n_estimators, max_depth=max_depth, 
                                 min_samples_split=min_samples_split, random_state=42, n_jobs=-1)
    
    X_sample = X_preprocessed.sample(20000, random_state=42)
    y_sample = y.loc[X_sample.index]
    return eval_pipeline(model, X_sample, y_sample)

In [ ]:
print("Tuning ExtraTreesClassifier (on subset)...")
study_et = optuna.create_study(direction='maximize')
study_et.optimize(objective_et, n_trials=N_TRIALS)
et_best_params = study_et.best_params
et_best_params.update({'random_state': 42, 'n_jobs': -1})

In [ ]:
# 2. MLPClassifier
def objective_mlp(trial):
    alpha = trial.suggest_float('alpha', 1e-5, 1e-1, log=True)
    hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', [(100,), (200,), (100, 50), (256, 128)])
    learning_rate_init = trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True)
    model = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, alpha=alpha, 
                          learning_rate_init=learning_rate_init, max_iter=200, random_state=42)
    
    X_sample = X_preprocessed.sample(15000, random_state=42)
    y_sample = y.loc[X_sample.index]
    return eval_pipeline(model, X_sample, y_sample)

In [ ]:
print("Tuning MLPClassifier (on subset)...")
study_mlp = optuna.create_study(direction='maximize')
study_mlp.optimize(objective_mlp, n_trials=N_TRIALS)
mlp_best_params = study_mlp.best_params
mlp_best_params.update({'max_iter': 200, 'random_state': 42})

In [ ]:
tuned_params_advanced = {
    'extratrees': et_best_params,
    'mlp': mlp_best_params
}

In [ ]:
with open('../datasets/tuned_parameters_advanced.json', 'w') as f:
    json.dump(tuned_params_advanced, f, indent=4)
print("Saved advanced tuned parameters.")

### OOF Generation

In [ ]:
print("Generating 5-Fold OOF Predictions with advanced models...")

In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

In [ ]:
oof_train_et = np.zeros(len(X_preprocessed))
test_preds_et = np.zeros(len(X_test_preprocessed))

In [ ]:
oof_train_mlp = np.zeros(len(X_preprocessed))
test_preds_mlp = np.zeros(len(X_test_preprocessed))

In [ ]:
oof_train_nb = np.zeros(len(X_preprocessed))
test_preds_nb = np.zeros(len(X_test_preprocessed))

In [ ]:
for fold, (train_idx, val_idx) in enumerate(cv.split(X_preprocessed, y)):
    print(f"--- Fold {fold + 1}/{N_FOLDS} ---")
    
    X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
    
    # Pre-fit preprocessor to avoid repetitive expensive imputation
    print("  Fitting Pipeline...")
    X_tr_prep = preprocessor.fit_transform(X_tr)
    X_val_prep = preprocessor.transform(X_val)
    X_test_prep = preprocessor.transform(X_test_preprocessed)
    
    # ExtraTrees
    clf_et = ExtraTreesClassifier(**et_best_params)
    clf_et.fit(X_tr_prep, y_tr)
    oof_train_et[val_idx] = clf_et.predict_proba(X_val_prep)[:, 1]
    test_preds_et += clf_et.predict_proba(X_test_prep)[:, 1] / N_FOLDS
    print(f"  ET Fold {fold+1} AUC: {roc_auc_score(y_val, oof_train_et[val_idx]):.5f}")
    
    # MLPClassifier
    clf_mlp = MLPClassifier(**mlp_best_params)
    clf_mlp.fit(X_tr_prep, y_tr)
    oof_train_mlp[val_idx] = clf_mlp.predict_proba(X_val_prep)[:, 1]
    test_preds_mlp += clf_mlp.predict_proba(X_test_prep)[:, 1] / N_FOLDS
    print(f"  MLP Fold {fold+1} AUC: {roc_auc_score(y_val, oof_train_mlp[val_idx]):.5f}")
    
    # GaussianNB
    clf_nb = GaussianNB()
    clf_nb.fit(X_tr_prep, y_tr)
    oof_train_nb[val_idx] = clf_nb.predict_proba(X_val_prep)[:, 1]
    test_preds_nb += clf_nb.predict_proba(X_test_prep)[:, 1] / N_FOLDS
    print(f"  NB Fold {fold+1} AUC: {roc_auc_score(y_val, oof_train_nb[val_idx]):.5f}")

In [ ]:
os.makedirs('../datasets/oof_preds', exist_ok=True)
np.save('../datasets/oof_preds/oof_train_et.npy', oof_train_et)
np.save('../datasets/oof_preds/test_preds_et.npy', test_preds_et)
np.save('../datasets/oof_preds/oof_train_mlp.npy', oof_train_mlp)
np.save('../datasets/oof_preds/test_preds_mlp.npy', test_preds_mlp)
np.save('../datasets/oof_preds/oof_train_nb.npy', oof_train_nb)
np.save('../datasets/oof_preds/test_preds_nb.npy', test_preds_nb)

In [ ]:
print("Saved all OOF and Test predictions successfully!")